# Module 11 · Solutions

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(11)
BASE = "data/"
uni = pd.read_csv(BASE+"nse_stock_universe.csv", parse_dates=["date"])
px = uni.pivot(index="date", columns="ticker", values="close").sort_index()
px.loc[:"2024-09-01", "TATAMOTORS.NS"] = px.loc[:"2024-09-01", "TATAMOTORS.NS"] / 5   # split fixed

def backtest(close, fast=50, slow=200, cost_bps=15):
    rets = close.pct_change()
    pos = (close.rolling(fast).mean() > close.rolling(slow).mean()).astype(int).shift(1)
    trades = pos.diff().abs().fillna(0)
    sr = pos*rets - trades*cost_bps/10_000
    eq_s = (1+sr.fillna(0)).cumprod(); eq_h = (1+rets.fillna(0)).cumprod()
    yrs = len(rets.dropna())/252
    return {"edge": eq_s.iloc[-1]**(1/yrs) - eq_h.iloc[-1]**(1/yrs), "trades": int(trades.sum()),
            "sr": sr, "rets": rets}

## 11A

In [ ]:
# Ex1 - the sell side
bids = pd.DataFrame({"price":[99.95,99.90,99.85,99.80,99.75], "qty":[400,900,1500,2200,3000]})
def market_sell(bids, qty):
    remaining, proceeds = qty, 0.0
    for _, row in bids.iterrows():
        take = min(remaining, row["qty"]); proceeds += take*row["price"]; remaining -= take
        if remaining <= 0: break
    return proceeds/qty
print(f"SELL 4,000 avg fill: Rs {market_sell(bids, 4000):.3f} (buy side was 100.147: symmetric here BY CONSTRUCTION)")
print("Real panics are asymmetric: bid depth evaporates (buyers cancel) while sellers stampede - impact")
print("on the way DOWN is famously worse. Our static book can't show fear; remember that it exists.")

# Ex2 - wider spreads
true_px = 100 + np.cumsum(rng.normal(0, 0.03, 500))
def run_maker(half_spread, p_hit):
    cash, inv, pnl = 0.0, 0, []
    for t in range(500):
        side = rng.choice(["b","s","n"], p=[p_hit, p_hit, 1-2*p_hit])
        if side=="b": cash += true_px[t]+half_spread; inv -= 1
        elif side=="s": cash -= true_px[t]-half_spread; inv += 1
        pnl.append(cash + inv*true_px[t])
    return pnl[-1], max(abs(min(0,min(np.minimum.accumulate([0])))), 0)
p1,_ = run_maker(0.05, 0.35); p2,_ = run_maker(0.15, 0.20)
print(f"\nNarrow&busy Rs {p1:.0f} vs wide&quiet Rs {p2:.0f} - the eternal dial: margin per trade vs volume.")
print("Real makers tune it continuously against volatility: wider when scared, tighter when confident.")

In [ ]:
# Ex3 - inventory limiter
cash, inv, pnl = 0.0, 0, []
for t in range(500):
    p_buy  = 0.35 if inv > -10 else 0.0      # stop quoting the side that grows the position
    p_sell = 0.35 if inv <  10 else 0.0
    side = rng.choice(["b","s","n"], p=[p_buy, p_sell, 1-p_buy-p_sell])
    if side=="b": cash += true_px[t]+0.05; inv -= 1
    elif side=="s": cash -= true_px[t]-0.05; inv += 1
    pnl.append(cash + inv*true_px[t])
print(f"With limiter: final P&L Rs {pnl[-1]:.0f}, inventory forever inside [-10, 10].")
print("Slightly less spread income, drastically less tail risk - the first risk control of every real system,")
print("and the shape of ALL of them: give up a little expected profit to cap the catastrophic state.")

## 11B

In [ ]:
# Ex1 - the parameter graveyard
results = {}
for fast, slow in [(20,100),(20,200),(50,100),(50,200)]:
    edges = [backtest(px[t].dropna(), fast, slow)["edge"] for t in px.columns]
    results[(fast,slow)] = np.median(edges)
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f"fast {k[0]:>3} / slow {k[1]:>3}: median edge {v*100:+.2f} pp")
print("\nThe best combo's number is CONTAMINATED: we chose it by looking at the full-sample results -")
print("the same test-set sin as 6B Ex1. Clean protocol: choose parameters on an early window (or other")
print("markets), then run ONCE on the held-out period and report THAT. Every grid search is a small look-ahead.")

In [ ]:
# Ex2 - the cost dial
for c in [5, 15, 40]:
    edges = [backtest(px[t].dropna(), cost_bps=c)["edge"] for t in px.columns]
    print(f"cost {c:>2} bps: median edge {np.median(edges)*100:+.2f} pp | positive on {np.mean([e>0 for e in edges]):.0%} of stocks")
print("\nBy ~40 bps the edge is dead everywhere. Who can run marginal strategies? Those whose costs")
print("approach zero: members, market makers (who EARN the spread others pay), and scale players.")
print("Retail pays the toll; the toll IS most of the house's edge.")

# Ex3 - regime cut
early, late = [], []
for t in px.columns:
    s = px[t].dropna()
    early.append(backtest(s[:"2023-12-31"])["edge"])
    late.append(backtest(s["2023-06-01":])["edge"])       # overlap so slow MA warms up
print(f"\nMedian edge 2022-23: {np.median(early)*100:+.2f} pp | 2024-25: {np.median(late)*100:+.2f} pp")
print("The edge (such as it is) is regime-dependent - trend rules earn in trending regimes and bleed in")
print("choppy ones. The Bias Check's fourth line was never decoration: EVERY strategy result is a regime claim.")